In [ ]:
%pip install ucimlrepo


: 

In [ ]:
from ucimlrepo import fetch_ucirepo
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Download dataset
bank_marketing = fetch_ucirepo(id=222)
 
# Get features and target
X = bank_marketing.data.features
y = bank_marketing.data.targets
 
print("Dataset loaded successfully!")

In [ ]:
print("First 5 rows:")
display(X.head())

# Dataset dimensions
print("Dataset Shape:", X.shape)

In [ ]:
# Information about columns
X.info()

In [ ]:
# Dataset description
display(X.describe())

In [ ]:
missing_count = X.isnull().sum()
missing_percentage = (missing_count / len(X)) * 100

missing_report = pd.DataFrame({
    "Missing Count": missing_count,
    "Missing Percentage": missing_percentage
})

display(missing_report[missing_report["Missing Count"] > 0])

In [ ]:
categorical_all = df.select_dtypes(include=["object", "str"]).columns

unknown_report = {}

for col in categorical_all:
    count = df[col].astype("string").str.lower().eq("unknown").sum()
    if count > 0:
        unknown_report[col] = int(count)

if unknown_report:
    display(pd.DataFrame.from_dict(
        unknown_report, orient="index", columns=["Unknown Count"]
    ))
else:
    print("No literal 'unknown' values found.")

In [ ]:
duplicate_count = df.duplicated().sum()

print("Duplicate rows:", duplicate_count)

In [ ]:
categorical_columns = X.select_dtypes(
    include=["object", "str"]
).columns

for column in categorical_columns:

    plt.figure(figsize=(8, 4))

    X[column].value_counts().head(10).plot(kind="bar")

    plt.title(column + " Distribution")
    plt.xlabel(column)
    plt.ylabel("Count")

    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

In [ ]:
numerical_columns = X.select_dtypes(
    include="number"
).columns

print("Numerical Columns:")
print(list(numerical_columns))

In [ ]:
X[numerical_columns].hist(
    figsize=(12, 8),
    bins=20
)

plt.suptitle("Numerical Features Distribution")

plt.tight_layout()
plt.show()

In [ ]:
for column in numerical_columns:

    plt.figure(figsize=(8, 3))

    sns.boxplot(x=X[column])

    plt.title(column + " Boxplot")

    plt.show()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [ ]:
# Remove duration column
X_model = X.drop(columns=["duration"], errors="ignore")
 
# Target variable
y_model = y.iloc[:, 0]
 
print("Features:", X_model.shape)
print("Target:", y_model.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_model,
    y_model,
    test_size=0.20,
    random_state=42,
    stratify=y_model
)
 
print("Training Data:", X_train.shape)
print("Testing Data:", X_test.shape)

In [ ]:
numeric_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns
 
categorical_features = X_train.select_dtypes(
    include=["object", "category", "bool"]
).columns
 
print("Numerical Features:")
print(list(numeric_features))
 
print("Categorical Features:")
print(list(categorical_features))

In [ ]:
# Numerical preprocessing
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
 
# Categorical preprocessing
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

In [ ]:
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])
 
# Fit on training data only
X_train_ready = preprocessor.fit_transform(X_train)
 
# Transform testing data
X_test_ready = preprocessor.transform(X_test)
 
print("Preprocessing Completed!")
 
print("Training Shape:", X_train_ready.shape)
print("Testing Shape:", X_test_ready.shape)

In [ ]:

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

import pandas as pd


In [ ]:

# Create Dummy Classifier
dummy_model = DummyClassifier(strategy="most_frequent")

# Train model
dummy_model.fit(X_train_ready, y_train)

# Make predictions
dummy_predictions = dummy_model.predict(X_test_ready)

print("Dummy Classifier Completed!")


In [ ]:

# Create Logistic Regression Model
logistic_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

# Train model
logistic_model.fit(X_train_ready, y_train)

# Make predictions
logistic_predictions = logistic_model.predict(X_test_ready)

print("Logistic Regression Completed!")


In [ ]:

def evaluate_model(name, y_true, predictions):

    accuracy = accuracy_score(y_true, predictions)

    precision = precision_score(
        y_true, predictions,
        pos_label="yes",
        zero_division=0
    )

    recall = recall_score(
        y_true, predictions,
        pos_label="yes",
        zero_division=0
    )

    f1 = f1_score(
        y_true, predictions,
        pos_label="yes",
        zero_division=0
    )

    return [name, accuracy, precision, recall, f1]


dummy_results = evaluate_model(
    "Dummy Classifier",
    y_test,
    dummy_predictions
)

logistic_results = evaluate_model(
    "Logistic Regression",
    y_test,
    logistic_predictions
)

results = pd.DataFrame(
    [dummy_results, logistic_results],
    columns=[
        "Model",
        "Accuracy",
        "Precision",
        "Recall",
        "F1-score"
    ]
)

display(results)


In [ ]:

print("Logistic Regression Results:")

print(classification_report(
    y_test,
    logistic_predictions,
    zero_division=0
))


In [ ]:

results.to_csv("baseline_results.csv", index=False)

print("Results saved successfully!")
